<a href="https://colab.research.google.com/github/KattaLasya/HPCLab/blob/main/HPC_lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup and Data Generation

In [ ]:
import numpy as np
import time
import multiprocessing as mp

In [ ]:
# Generate a large dataset
N = 10_000_000  # 10 million elements
data = np.random.rand(N).astype(np.float64)

print("Dataset size:", data.size)

Dataset size: 10000000


Serial Reduction

In [ ]:
def serial_sum_max(arr):
    total_sum = 0.0
    max_val = arr[0]
    for x in arr:
        total_sum += x
        if x > max_val:
            max_val = x
    return total_sum, max_val

In [ ]:
start = time.time()
serial_result = serial_sum_max(data)
serial_time = time.time() - start

print("Serial Result:", serial_result)
print("Serial Time:", serial_time, "seconds")


Serial Result: (np.float64(4999613.471587877), np.float64(0.9999998159284245))
Serial Time: 1.9826176166534424 seconds


Parallel Reduction

In [ ]:
def partial_sum_max(chunk):
    return np.sum(chunk), np.max(chunk)

In [ ]:
def parallel_sum_max(arr, num_processes):
    chunks = np.array_split(arr, num_processes)

    with mp.Pool(processes=num_processes) as pool:
        results = pool.map(partial_sum_max, chunks)

    total_sum = sum(r[0] for r in results)
    max_val = max(r[1] for r in results)

    return total_sum, max_val

In [ ]:
num_processes = mp.cpu_count()

start = time.time()
parallel_result = parallel_sum_max(data, num_processes)
parallel_time = time.time() - start

print("Parallel Result:", parallel_result)
print("Parallel Time:", parallel_time, "seconds")
print("CPU Cores Used:", num_processes)

Parallel Result: (np.float64(4999613.471588198), np.float64(0.9999998159284245))
Parallel Time: 0.37866663932800293 seconds
CPU Cores Used: 2


Numpy Reduction

In [ ]:
start = time.time()
numpy_sum = np.sum(data)
numpy_max = np.max(data)
numpy_time = time.time() - start

print("NumPy Result:", (numpy_sum, numpy_max))
print("NumPy Time:", numpy_time, "seconds")

NumPy Result: (np.float64(4999613.471588199), np.float64(0.9999998159284245))
NumPy Time: 0.04950833320617676 seconds


Correctness Verification

In [ ]:
def check_correctness(ref, test, tol=1e-6):
    return abs(ref[0] - test[0]) < tol and abs(ref[1] - test[1]) < tol

print("Serial vs Parallel:", check_correctness(serial_result, parallel_result))
print("Serial vs NumPy:", check_correctness(serial_result, (numpy_sum, numpy_max)))

Serial vs Parallel: True
Serial vs NumPy: True


performance Summary

In [ ]:
print("\nPerformance Summary")
print("-------------------")
print(f"Serial Time   : {serial_time:.4f} s")
print(f"Parallel Time : {parallel_time:.4f} s")
print(f"NumPy Time    : {numpy_time:.4f} s")
print(f"Speedup (Parallel vs Serial): {serial_time / parallel_time:.2f}x")


Performance Summary
-------------------
Serial Time   : 1.9826 s
Parallel Time : 0.3787 s
NumPy Time    : 0.0495 s
Speedup (Parallel vs Serial): 5.24x
